# Phase 7 — Final Training Configuration

Transfers official YOLO11s pretrained weights into the project CBAM architecture, verifies the current final dataset, benchmarks the image-size decision, and reports the final recommended configuration. It does not start training or run epochs.

In [1]:
from pathlib import Path
import json
import math
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import torch
import yaml
from ultralytics import __version__ as ultralytics_version
from ultralytics.data.dataset import YOLODataset
from weapon_threat_detection.artifacts import configure_logger, utc_timestamp, write_json
from weapon_threat_detection.config import load_config
from weapon_threat_detection.dataset_review import audit_duplicate_annotations
from weapon_threat_detection.engineering import MERGED_CLASS_NAMES, collect_dataset_statistics
from weapon_threat_detection.model_engineering import benchmark_memory, detect_hardware, load_yaml, serialize_hardware
from weapon_threat_detection.transfer_learning import freeze_transferred_backbone_layers, load_pretrained_weights, serialize_transfer_report
from weapon_threat_detection.validation import summarize_records, validate_dataset

config = load_config(ROOT / 'configs' / 'project.yaml')
training = load_yaml(ROOT / 'configs' / 'training.yaml')
augmentation = load_yaml(ROOT / 'configs' / 'final_augmentation_config.yaml')['augmentation']
experiment = load_yaml(ROOT / 'configs' / 'experiment.yaml')['experiment']
dataset_root = ROOT / experiment['final_dataset']
reports_dir = ROOT / experiment['reports_directory']
logger = configure_logger(ROOT / experiment['logs_directory'], 'final_training_configuration')
hardware = detect_hardware()
model, transfer = load_pretrained_weights(ROOT / training['training']['model'], ROOT / 'configs' / 'training.yaml', ROOT / training['training']['pretrained_checkpoint'], nc=len(MERGED_CLASS_NAMES))
transfer_report = serialize_transfer_report(transfer)
dataset_records = validate_dataset(dataset_root, config['dataset']['splits'], len(MERGED_CLASS_NAMES), config['dataset']['image_extensions'])
validation_summary = summarize_records(dataset_records)
duplicates = audit_duplicate_annotations(dataset_root, config['dataset']['splits'])
if duplicates['duplicate_annotations'] != 0 or any(status not in {'valid', 'valid_background'} for status in validation_summary):
    raise RuntimeError({'duplicates': duplicates['duplicate_annotations'], 'validation': validation_summary})
statistics = collect_dataset_statistics(dataset_root, MERGED_CLASS_NAMES, config['dataset']['splits'])
benchmark = benchmark_memory(model, hardware.device, training['training']['image_size'], [24, 28, 32], iterations=1, warmup_iterations=0)
limit_mb = hardware.total_memory_gb * 1024 * 0.95
safe = [record for record in benchmark if record['status'] == 'stable' and record['peak_memory_mb'] < limit_mb]
maximum_safe_batch = max(safe, key=lambda record: record['batch_size'])
if maximum_safe_batch['batch_size'] != training['training']['batch_size']:
    raise RuntimeError({'configured_batch_size': training['training']['batch_size'], 'maximum_safe_batch': maximum_safe_batch})
freeze = freeze_transferred_backbone_layers(model, training['training']['freeze_through_layer'])
data = yaml.safe_load((dataset_root / 'data.yaml').read_text(encoding='utf-8'))
dataset = YOLODataset(img_path=str(dataset_root / 'train/images'), imgsz=training['training']['image_size'], augment=False, cache=False, data=data)
batch = dataset.collate_fn([dataset[0], dataset[1]])
batch = {key: (value.to(hardware.device) if isinstance(value, torch.Tensor) else value) for key, value in batch.items()}
batch['img'] = batch['img'].float() / 255
model = model.to(hardware.device).train()
outputs = model(batch['img'])
loss, components = model(batch)
loss.sum().backward()
model.zero_grad(set_to_none=True)
smoke_test = {'batch_size': int(batch['img'].shape[0]), 'forward_output_keys': list(outputs), 'loss': float(loss.sum().detach().cpu()), 'loss_components': {key: float(value.detach().cpu()) for key, value in components.items()}, 'runtime_errors': 0}
steps_per_epoch = math.ceil(statistics['splits']['train']['images'] / training['training']['batch_size'])
time_estimation = {'steps_per_epoch': steps_per_epoch, 'full_unfreeze_images_per_second': maximum_safe_batch['images_per_second'], 'training_only_minutes_per_epoch': statistics['splits']['train']['images'] / maximum_safe_batch['images_per_second'] / 60, 'training_only_hours_for_configured_epochs': statistics['splits']['train']['images'] / maximum_safe_batch['images_per_second'] * training['training']['epochs'] / 3600, 'qualification': 'Measured synthetic forward/backward throughput excludes data loading, validation, checkpoints, and startup overhead.'}
report = {'dataset': {'path': str(dataset_root), 'splits': statistics['splits'], 'total_images': statistics['total_images'], 'background_images': statistics['background_images'], 'class_distribution': statistics['annotations_per_class'], 'classes': data['names']}, 'model': {'architecture': 'YOLO11s with CBAM', 'cbam_locations': ['P3', 'P4', 'P5'], 'parameters': transfer_report['total_model_parameters'], 'ultralytics_version': ultralytics_version, 'pretrained_checkpoint': training['training']['pretrained_checkpoint'], 'transfer_coverage_percent': transfer_report['transferred_percentage']}, 'training': {**training['training'], 'lr_fraction': 'cosine scheduler-managed', 'maximum_safe_800_batch': maximum_safe_batch, 'full_unfreeze_benchmark': benchmark}, 'loss': training['loss'], 'augmentation': augmentation, 'hardware': serialize_hardware(hardware), 'freeze_plan': {'first_10_epochs': 'Freeze model layers 0 through 10, including CBAM modules at P3/P4.', 'remaining_epochs': 'Unfreeze all model layers for epochs 11 through 80.', 'verification': freeze}, 'smoke_test': smoke_test, 'training_time_estimation': time_estimation, 'training_execution_plan': 'Training has not started. The configuration is verified and frozen pending explicit approval for the one final experiment.', 'ready_for_final_training': True, 'training_started': False}
stamp = utc_timestamp()
report_path = write_json(reports_dir / ('final_training_configuration_' + stamp + '.json'), report)
logger.info('Final configuration report generated: %s', report_path)
print({'configuration_report': str(report_path), 'maximum_safe_800_batch': maximum_safe_batch, 'training_started': False})

Fast image access ✅ (ping: 0.0±0.0 ms, read: 224.1±111.5 MB/s, size: 100.7 KB)



Scanning /Users/mohamedtarek/weapon130/processed/final_dataset_v1/train/labels.cache... 105475 images, 9032 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 105475/105475 224.5Mit/s 0.0s

{'configuration_report': '/Users/mohamedtarek/weapon130/reports/final_training_configuration_20260723T112712Z.json', 'maximum_safe_800_batch': {'batch_size': 28, 'status': 'stable', 'peak_memory_mb': 21421.796875, 'images_per_second': 1.7631464375942372, 'error': None}, 'training_started': False}
